## Imports

In [4]:
%load_ext autoreload
%autoreload 2

from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import pylab as pl
import pandas as pd
import numpy as np
import scipy as sp
from scipy import stats
import yaml
import shelve

from caa_model.config import *
from caa_model.utils.get_yaml_data import ERStruct, RTConf, reformat_revised_data
from caa_model.models import base_model, disp_classic, sisp_classic, sisp_var_rbound, sisp_z0_var_rbound, disp_staggered
from caa_model.models.results import *
from caa_model.models.base_model import DATA
from caa_model.models.disp_classic import DISPClassicDDM
from caa_model.models.sisp_classic import SISPClassicDDM
from caa_model.models.sisp_var_rbound import SISPVarRBoundDDM
from caa_model.models.sisp_z0_var_rbound import SISPZ0VarRBoundDDM
from caa_model.models.disp_staggered import DISPStaggeredDDM

## Setup

In [5]:
caa_cfg = CAAConfig(
    delta_t=0.025,
    nr_ssteps=8192,
    nr_samples=10000,
    max_t=10.0,
)

db = shelve.open("data/mengxue_data.yml.dat", "r")
data = db["empirical_results"]
db.close()

## DISP Classic vs SISP Z0 Var R Bound

In [ ]:
# Instantiate both models (assuming DISP is defined similarly in your workspace)
sisp_model = SISPZ0VarRBoundDDM(config=caa_cfg, use_fftw=True, nr_threads=1)
disp_model = DISPClassicDDM(config=caa_cfg, use_fftw=True, nr_threads=1)

# (Make sure to define disp_param_bounds in your script)
sisp_param_bounds = sisp_z0_var_rbound.param_bounds
disp_param_bounds = disp_classic.param_bounds

# 2. Define Data Filter
def get_subject_data(full_data, subject_id):
    subject_id = str(subject_id).strip()
    
    # Order must match ERStruct: ["know_hit", "rem_hit", "know_fa", "rem_fa", "CR", "miss"]
    categories = ["know_hit", "rem_hit", "know_fa", "rem_fa", "CR", "miss"]
    filtered_cats = []

    for category in categories:
        cat_data = getattr(full_data, category)
        mask = cat_data.subj == subject_id

        filtered_cat = RTConf(
            rt=cat_data.rt[mask],
            conf=cat_data.conf[mask],
            target=cat_data.target[mask],
            subj=cat_data.subj[mask],
        )
        filtered_cats.append(filtered_cat)

    # Returning top-level ERStruct makes it fully picklable for parallel workers
    return ERStruct(*filtered_cats)

# Dynamically grab unique subjects
subject_ids = np.unique(data.rem_hit.subj) 
print(f"Found {len(subject_ids)} subjects to fit.")

# 3. Main Fitting Loop
results = []

for subj in subject_ids:
    print(f"\n--- Fitting Subject {subj} ---")

    subj_data = get_subject_data(data, subj)

    # SISP Fit & Evaluate
    print("Fitting SISP...")
    sisp_params = sisp_model.fit(sisp_param_bounds, data=subj_data, method="qmpe", nr_workers=-1)
    sisp_eval = sisp_model.evaluate(sisp_params, data=subj_data, method="qmpe")

    # DISP Fit & Evaluate
    print("Fitting DISP...")
    disp_params = disp_model.fit(disp_param_bounds, data=subj_data, method="qmpe", nr_workers=-1)
    disp_eval = disp_model.evaluate(disp_params, data=subj_data, method="qmpe")

    sisp_won = sisp_eval.bic < disp_eval.bic

    row_data = {
        "subject": subj,
        "trials": sisp_eval.n_trials,
        "sisp_nll": sisp_eval.nll,
        "disp_nll": disp_eval.nll,
        "sisp_bic": sisp_eval.bic,
        "disp_bic": disp_eval.bic,
        "bic_diff": disp_eval.bic - sisp_eval.bic, 
        "sisp_won": sisp_won,
    }
    
    # Flatten the parameters into the row with prefixes to keep them distinct
    row_data.update({f"sisp_{k}": v for k, v in sisp_params._asdict().items()})
    row_data.update({f"disp_{k}": v for k, v in disp_params._asdict().items()})

    results.append(row_data)

    # Autosave to prevent catastrophic data loss
    pd.DataFrame(results).to_csv("results/autosave_disp_classic_vs_sisp_z0_var_rbound.csv", index=False)
    print(f"Subject {subj} finished. SISP Won: {sisp_won}. Progress autosaved.")

# 4. Generate Final Win-Rate Report
results_df = pd.DataFrame(results)
total_subjects = len(results_df)
sisp_wins = results_df["sisp_won"].sum()

print(f"\n--- SHOWDOWN RESULTS ---")
print(f"SISP won {sisp_wins} out of {total_subjects} subjects ({(sisp_wins/total_subjects)*100:.1f}%).")
print(f"Average BIC Difference (DISP - SISP): {results_df['bic_diff'].mean():.2f}")

Found 4 subjects to fit.

--- Fitting Subject 1 ---
Fitting SISP...
differential_evolution step 1: f(x)= 1196.1403755244596
differential_evolution step 2: f(x)= 1196.1403755244596
differential_evolution step 3: f(x)= 1139.7511647847355
differential_evolution step 4: f(x)= 1139.7511647847355
differential_evolution step 5: f(x)= 549.0748783322215
differential_evolution step 6: f(x)= 549.0748783322215
differential_evolution step 7: f(x)= 549.0748783322215
differential_evolution step 8: f(x)= 549.0748783322215
differential_evolution step 9: f(x)= 549.0748783322215
differential_evolution step 10: f(x)= 549.0748783322215
differential_evolution step 11: f(x)= 497.24374335130005
differential_evolution step 12: f(x)= 497.24374335130005
differential_evolution step 13: f(x)= 452.9287762174654
differential_evolution step 14: f(x)= 352.2524389397918
differential_evolution step 15: f(x)= 299.55726074382255
differential_evolution step 16: f(x)= 299.55726074382255
differential_evolution step 17: f(x)=